# Project - Deploy and Test a Multi-User Conversational Chatbot API with LangServe

## Install OpenAI, and LangChain dependencies

Install the following httpx library version for compatibility with other libraries

In [ ]:
!pip install httpx==0.27.2

In [1]:
!pip install langchain==0.2.0
!pip install langchain-openai==0.1.7
!pip install langchain-community==0.2.0
!pip install langserve[all]==0.2.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 973.7/973.7 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.8/332.8 kB 18.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.0/145.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.5/327.5 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 21.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Enter Open AI API Key

In [2]:
from getpass import getpass

OPENAI_KEY = getpass('Enter Open AI API Key: ')

Enter Open AI API Key: ··········


## Setup Environment Variables

In [3]:
import os

os.environ['OPENAI_API_KEY'] = OPENAI_KEY

In [4]:
# save server API into a python file to be deployed
%%writefile langchain_server_api.py
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from langchain.memory import FileChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from pathlib import Path
from langserve import add_routes
import os

# Create an instance of FastAPI to serve as the main application.
app = FastAPI(
    title="LangChain Server",
    version="1.0",
    description="Spin up a simple API server using Langchain's Runnable interfaces",
)

# Configure CORS middleware to allow all origins, enabling cross-origin requests.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
    expose_headers=["*"],
)

@app.get("/liveness")
def liveness():
    """
    Define a liveness check endpoint.

    This route is used to verify that the API is operational and responding to requests.

    Returns:
        A simple string message indicating the API is working.
    """
    return 'API Works!'

### BUILD THE CONVERSATION CHAIN ###

# Configure the chat prompt template and define the conversation chain.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Act as a helpful AI Assistant"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{human_input}"),
    ]
)

# Initialize the OpenAI Chat instance with specific model parameters.
chatgpt = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

# create simple llm chain
llm_chain = (prompt
                |
             chatgpt
                |
             StrOutputParser()
)

# create store for managing individual user session history
# create directory to store user conversations
history_dir = Path("./message_history")
history_dir.mkdir(parents=True, exist_ok=True)
# Define a function to retrieve the message history for a given user session id.
def get_session_history(session_id: str) -> FileChatMessageHistory:
    session_history_path = history_dir / f"{session_id}.json"
    return FileChatMessageHistory(str(session_history_path))

# Wrap the conversation chain in a RunnableWithMessageHistory to handle session-based history.
conv_chain = RunnableWithMessageHistory(
    llm_chain,
    get_session_history,
    input_messages_key="human_input",
    history_messages_key="history",
)


# Register routes using LangChain's utility function which integrates the chat model into the API.
add_routes(
    app,
    conv_chain,
    path="/chatbot",
)

if __name__ == "__main__":
    import uvicorn
    # Start the server on localhost at port 8585.
    uvicorn.run(app, host="127.0.0.1", port=8585)

Writing langchain_server_api.py


In [5]:
!python langchain_server_api.py &>./app_logs.txt &

In [6]:
!ps -ef | grep langchain_server_api

root        2208       1 36 21:29 ?        00:00:02 python3 langchain_server_api.py
root        2239     142  0 21:29 ?        00:00:00 /bin/bash -c ps -ef | grep langchain_server_api
root        2241    2239  0 21:29 ?        00:00:00 grep langchain_server_api


In [ ]:
# IGNORE THIS: only use this to kill your app if needed with a process id
!sudo kill -9 7865

## Load Dependencies

In [7]:
import requests

## Check if API works

In [8]:
response = requests.get('http://127.0.0.1:8585/liveness')

In [9]:
response.json(), response.status_code

('API Works!', 200)

## Connect to the LLM API endpoint

In [10]:
from langserve import RemoteRunnable

chat_endpoint = RemoteRunnable("http://localhost:8585/chatbot")
def chat_with_llm(prompt: str, session_id: str):
    for chunk in chat_endpoint.stream({"human_input": prompt},
                                      {'configurable': { 'session_id': session_id}}):
        print(chunk, end="", flush=True)

### Chat as User 1

In [11]:
user_id = 'bob123'
prompt = "Hi I am Bob, can you explain AI in 3 bullet points?"
chat_with_llm(prompt, user_id)

- AI, or artificial intelligence, refers to the simulation of human intelligence processes by machines, such as learning, reasoning, and problem-solving.
- AI technologies include machine learning, natural language processing, computer vision, and robotics, among others.
- AI is used in various industries and applications, such as healthcare, finance, autonomous vehicles, and virtual assistants, to improve efficiency, accuracy, and decision-making.

In [12]:
prompt = "Now do the same for Generative AI"
chat_with_llm(prompt, user_id)

- Generative AI is a subset of artificial intelligence that focuses on creating new content, such as images, text, or music, rather than analyzing existing data.
- Generative AI models, such as Generative Adversarial Networks (GANs) and Variational Autoencoders (VAEs), can generate realistic and novel outputs by learning patterns and structures from training data.
- Generative AI is used in creative applications, such as art generation, content creation, and design, as well as in fields like healthcare and drug discovery for generating new molecules and compounds.

In [13]:
prompt = "What about deep learning?"
chat_with_llm(prompt, user_id)

- Deep learning is a subset of machine learning that uses artificial neural networks with multiple layers to learn and extract patterns from large amounts of data.
- Deep learning models, such as convolutional neural networks (CNNs) and recurrent neural networks (RNNs), can automatically discover and represent complex features in data for tasks like image recognition, speech recognition, and natural language processing.
- Deep learning has achieved remarkable success in various domains, including computer vision, speech recognition, autonomous driving, and healthcare, by enabling machines to learn and make decisions without explicit programming.

In [14]:
prompt = "Mention in bullet points whatever we have discussed so far briefly"
chat_with_llm(prompt, user_id)

- AI refers to the simulation of human intelligence processes by machines, such as learning, reasoning, and problem-solving.
- Generative AI focuses on creating new content, like images and text, using models such as GANs and VAEs.
- Deep learning is a subset of machine learning that uses neural networks with multiple layers to learn patterns from data for tasks like image recognition and natural language processing.

## Chat as user 2

In [15]:
user_id = 'jim321'
prompt = "Hi I am Jim, can you tell me what is a large language model in 3 bullet points?"
chat_with_llm(prompt, user_id)

- A large language model is a type of artificial intelligence system that is trained on vast amounts of text data to understand and generate human language.
- These models are capable of performing a wide range of natural language processing tasks, such as text generation, translation, summarization, and more.
- Examples of large language models include GPT-3 (Generative Pre-trained Transformer 3) developed by OpenAI and BERT (Bidirectional Encoder Representations from Transformers) developed by Google.

In [16]:
prompt = "What on earth is a BERT model?"
chat_with_llm(prompt, user_id)

BERT (Bidirectional Encoder Representations from Transformers) is a large language model developed by Google that has significantly advanced the field of natural language processing (NLP). Here are some key points about BERT:

- BERT is a transformer-based model that is pre-trained on a large corpus of text data to understand the context and relationships between words in a sentence.
- Unlike previous language models that process text in a unidirectional manner, BERT is bidirectional, meaning it can consider the entire context of a word by looking at both the left and right context in a sentence.
- BERT has been shown to achieve state-of-the-art performance on a wide range of NLP tasks, such as question answering, sentiment analysis, and text classification, by fine-tuning the pre-trained model on specific datasets.

In [17]:
prompt = "What about this GPT-3 model you mentioned?"
chat_with_llm(prompt, user_id)

GPT-3 (Generative Pre-trained Transformer 3) is a large language model developed by OpenAI that has gained significant attention for its impressive capabilities in natural language processing. Here are some key points about GPT-3:

- GPT-3 is the third iteration of the Generative Pre-trained Transformer series, which uses a transformer architecture to process and generate human-like text.
- With 175 billion parameters, GPT-3 is one of the largest language models ever created, allowing it to generate coherent and contextually relevant text across a wide range of tasks.
- GPT-3 has demonstrated strong performance on various NLP tasks, such as text completion, translation, summarization, and more, without the need for task-specific training data. Its ability to generate human-like text has sparked interest and debate about the potential applications and implications of such advanced AI models.

In [18]:
prompt = "I am very forgetful, can you remind me what we discussed briefly in bullet points?"
chat_with_llm(prompt, user_id)

- We discussed what a large language model is, highlighting its role in understanding and generating human language through vast text data training.
- We explored BERT (Bidirectional Encoder Representations from Transformers), a transformer-based model developed by Google for advanced natural language processing tasks.
- We also covered GPT-3 (Generative Pre-trained Transformer 3), a large language model by OpenAI known for its impressive capabilities in generating human-like text across various NLP tasks.

Changing user id uses conversation only relevant to that user

In [19]:
user_id = 'bob123'
prompt = "Hey ChatGPT I don't remember what we discussed earlier, please tell me briefly in bullets?"
chat_with_llm(prompt, user_id)

- AI: Simulation of human intelligence processes by machines for learning, reasoning, and problem-solving.
- Generative AI: Subset of AI creating new content like images and text using models such as GANs and VAEs.
- Deep Learning: Subset of machine learning using neural networks with multiple layers to learn patterns from data for tasks like image recognition and natural language processing.